In [14]:
# 학번과 이름을 출력하세요
# 예시
# 학번 = '30261240'
# 이름 = '이현정'

학번 = '20251773'
이름 = '하사르군'

print(학번, 이름)

20251773 하사르군


In [27]:


promos = [
    globals()[name]
    for name in globals()
    if name.endswith('_promo')
    and name != 'best_promo'
]

print(promos)

[<function fidelity_promo at 0x1121a0fe0>, <function bulk_item_promo at 0x1121a13a0>, <function large_order_promo at 0x1121a1300>]


In [28]:
def fidelity_promo(order):
    """충성도 점수가 1000점 이상인 고객에게 전체 5% 할인 적용"""
    return order.total() * 0.05 if order.customer.fidelity >= 1000 else 0


def bulk_item_promo(order):
    """20개 이상의 동일 상품을 구입하면 10% 할인 적용"""
    discount = 0

    for item in order.cart:
        if item.quantity >= 20:
            discount += item.total() * 0.1

    return discount


def large_order_promo(order):
    """10종류 이상의 상품을 구입하면 전체 7% 할인 적용"""
    distinct_items = {item.product for item in order.cart}

    if len(distinct_items) >= 10:
        return order.total() * 0.07

    return 0

In [23]:
class Order:  # Context

    

    def __init__(self, customer, cart, promotion=None):
        self.customer = customer
        self.cart = list(cart)
        self.promotion = promotion  # 할인 함수

    def total(self):

        if not hasattr(self, '_total'):
            self._total = sum(item.total() for item in self.cart)

        return self._total

    def due(self):

        if self.promotion is None:
            discount = 0
        else:
            discount = self.promotion(self)  # promotion은 함수 객체

        return self.total() - discount

    def __repr__(self):

        fmt = '<Order total: {:.2f} due: {:.2f}>'

        return fmt.format(self.total(), self.due())

In [ ]:
class Order:  # Context

    """고객(namedtuple) 및 LineItem 클래스의 인스턴스들을
    cart로 받아서 총 계산할 가격을 산출
    """

    def __init__(self, customer, cart, promotion=None):
        self.customer = customer
        self.cart = list(cart)
        self.promotion = promotion  # 할인 함수

    def total(self):

        if not hasattr(self, '_total'):
            self._total = sum(item.total() for item in self.cart)

        return self._total

    def due(self):

        if self.promotion is None:
            discount = 0
        else:
            discount = self.promotion(self)  # promotion은 함수 객체

        return self.total() - discount

    def __repr__(self):

        fmt = '<Order total: {:.2f} due: {:.2f}>'

        return fmt.format(self.total(), self.due())

In [8]:
joe = Customer('John Doe', 0)
ann = Customer('Ann Smith', 1100)

cart = [
    LineItem('banana', 4, .5),
    LineItem('apple', 10, 1.5),
    LineItem('watermelon', 5, 5.0)
]

In [ ]:
class FidelityPromo(Promotion):

    """충성도 점수가 1000점 이상인 고객에게 전체 5% 할인 적용"""

    def discount(self, order):
        return order.total() * 0.05 if order.customer.fidelity >= 1000 else 0


class BulkItemPromo(Promotion):

    """20개 이상의 동일 상품을 구입하면 10% 할인 적용"""

    def discount(self, order):
        discount = 0

        for item in order.cart:
            if item.quantity >= 20:
                discount += item.total() * 0.1

        return discount


class LargeOrderPromo(Promotion):

    """10종류 이상의 상품을 구입하면 전체 7% 할인 적용"""

    def discount(self, order):
        distinct_items = {item.product for item in order.cart}

        if len(distinct_items) >= 10:
            return order.total() * 0.07

        return 0

In [6]:
class Promotion(ABC):  # Strategy: abstract base class

    """할인 혜택 클래스들의 형태를 선언"""

    @abstractmethod
    def discount(self, order):
        """할인액을 구체적인 숫자로 반환"""
        pass

In [ ]:
class Order:  # Context

    """고객(Customer) 및 LineItem 클래스의 인스턴스들을
    cart로 받아서 총 계산할 가격을 산출
    """

    def __init__(self, customer, cart, promotion=None):
        self.customer = customer
        self.cart = list(cart)
        self.promotion = promotion  # 할인 객체

    def total(self):
        """_total 속성이 없으면 전체 계산할 값을 계산"""
        if not hasattr(self, '_total'):
            self._total = sum(item.total() for item in self.cart)

        return self._total

    def due(self):
        """할인금액 차감"""
        if self.promotion is None:
            discount = 0
        else:
            discount = self.promotion.discount(self)  # self = Order 객체

        return self.total() - discount

    def __repr__(self):
        fmt = '<Order total: {:.2f} due: {:.2f}>'
        return fmt.format(self.total(), self.due())

In [3]:
from abc import ABC, abstractmethod
from collections import namedtuple

Customer = namedtuple('Customer', 'name fidelity')

park = Customer('Park', 100)

print(park)


class LineItem:
    """구매할 물품/갯수를 생성해서 총 가격을 반환"""

    def __init__(self, product, quantity, price):
        self.product = product
        self.quantity = quantity
        self.price = price

    def total(self):
        return self.price * self.quantity

Customer(name='Park', fidelity=100)
